# 01 — Data and spectral EDA

**Question this notebook answers:** what *is* this data, physically, and what
makes the ten land-cover classes separable?

Satellite imagery is not a picture. Each pixel is a set of calibrated
measurements of reflected sunlight in thirteen wavelength bands, and almost
everything that follows in this project — which classes confuse, which bands
matter, why a vision-language model struggles — is already visible in those
measurements. So this chapter is deliberately heavy on the physics before any
model appears.

**Produces**
- `outputs/splits.npz` — the stratified 70/15/15 split every later notebook uses
- `outputs/band_stats.json` — per-band mean/std computed on the **training split only**
- `figures/01_*.png` — class distribution, stretch comparison, composites,
  spectral signatures, index boxplots

**Expected runtime:** ~10 minutes on a Colab T4 (dominated by the one-off
EuroSAT download and conversion; ~2 minutes on a re-run from cache).

### Environment setup and persistence

On Colab this clones the repository, installs the pinned dependencies, and — the
part that matters — mounts Google Drive so that **outputs survive the session**.
Locally it is a no-op beyond putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the split and normalisation
statistics, 02 writes the model checkpoint that 04, 05 and 06 all load. Without
persistence, a disconnect means re-running earlier chapters. So `outputs/` and
`figures/` are redirected to Drive via environment variables read by
`s2map.config` at import time.

**`data/` is deliberately NOT on Drive.** The EuroSAT cache is ~2.9 GB and every
training epoch reads it randomly; Drive is a network mount and random reads
through it are slow enough to dominate the runtime. Re-downloading into the
local VM disk each session costs a few minutes and is the faster trade. Set
`USE_DRIVE = False` to keep everything ephemeral.

The install is wrapped so a fragile wheel prints a clear message instead of
killing the kernel halfway through a 40-minute session.

**If this cell reports a numpy mismatch**, restart the runtime
(*Runtime → Restart session*) and run it again. Colab preinstalls a mutually
consistent numpy/torch/scipy set; when pip replaces one of them on disk, the
kernel is left holding the old version in memory and every compiled extension
raises `ValueError: numpy.dtype size changed`. Only a restart fixes it. The
requirements file uses compatible *ranges* rather than exact pins for exactly
this group of packages, so it should not happen — the check is there because it
is the single most common way a Colab notebook dies.

In [ ]:
# --- edit these two if you are running your own fork -----------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "s2-chips-to-map"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/s2-chips-to-map")
        for sub in ("outputs", "figures"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        # Read by s2map.config at import time, so this must happen before the import below.
        os.environ["S2MAP_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["S2MAP_FIGURE_DIR"] = str(PERSIST / "figures")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
        # Pick up fixes without needing a fresh VM: a stale clone from an earlier
        # session is otherwise invisible and confusing.
        subprocess.run(["git", "pull", "--ff-only", "--quiet"], check=False)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway — the cells below will report what is missing")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from s2map import config as C
C.ensure_dirs()
C.print_environment()
print()
# Catches the one Colab failure mode that no amount of re-running fixes:
# pip replaced numpy on disk after this kernel already imported it.
C.check_environment()
cfg = C.load_config()
SEED = C.set_seed(cfg.seed)
print(f"seed             {SEED}   (multi-seed runs use {cfg.seeds})")
DEVICE = C.get_device()
print(f"device           {DEVICE}")
print(f"outputs ->       {C.OUTPUT_DIR}")
print(f"figures ->       {C.FIGURE_DIR}")
print(f"data cache ->    {C.DATA_DIR}  (local/ephemeral by design — see the note above)")
if DEVICE == "cpu":
    print("\n!! no GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.")

### Load EuroSAT — the 13-band version

EuroSAT is 27,000 chips of 64×64 pixels cut out of Sentinel-2 scenes over 34
European countries, each labelled with a single land-cover class. It exists in
two variants and the choice matters: the widely used RGB variant is a JPEG
rendering with three channels, while the multispectral variant keeps all
thirteen original bands. **We use the 13-band version**, because ten of those
bands — red edge, near-infrared, shortwave infrared — carry exactly the
information that makes land cover separable and that an RGB photograph model
cannot see. That contrast is the spine of this project.

`s2map.data.load_eurosat_ms` tries, in order: an existing local cache, a local
extracted GeoTIFF tree, torchgeo's loader with auto-download, then the
HuggingFace mirrors. Whichever path succeeds is printed, along with everything
that was tried and failed — no silent fallbacks.

In [ ]:
import numpy as np
from s2map import data as D

bundle = D.load_eurosat_ms()

print("acquisition log:")
for note in bundle.notes:
    print("  -", note)
print()
print(bundle.describe())
print()
print(f"value range      min={int(np.asarray(bundle.X[:200]).min())} "
      f"max={int(np.asarray(bundle.X[:200]).max())}  (sampled from 200 chips)")

X, y = bundle.X, bundle.y
assert X.shape[1:] == (13, 64, 64), X.shape
assert y.shape[0] == X.shape[0]

### The sensor: what the thirteen numbers per pixel actually are

Sentinel-2 carries the MultiSpectral Instrument, which samples the reflected
solar spectrum in thirteen bands from 443 nm (deep blue / aerosol) to 2190 nm
(shortwave infrared). Each band is chosen for a physical purpose, not for
prettiness — B08 sits in the near-infrared because leaf mesophyll scatters
strongly there while chlorophyll absorbs red, so the B04→B08 contrast is a
direct measure of green biomass.

In [ ]:
from IPython.display import Markdown, display
from s2map import bands as B

display(Markdown(B.band_table_markdown()))
print("channel order used everywhere in this repo:")
print(" ", " ".join(f"{i}:{b}" for i, b in enumerate(B.BAND_IDS)))

### Three consequences of the sensor that shape everything downstream

**1. The native resolutions differ, and EuroSAT hides that.** Four bands are
10 m, six are 20 m, three are 60 m. EuroSAT has already resampled all thirteen
onto a common 10 m / 64×64 grid, so the differences are invisible here. That is
convenient and slightly misleading: in notebook 05, where we read a real scene,
the resampling has to be done explicitly and the choice of interpolation is a
decision we have to defend.

**2. EuroSAT is Level-1C — top-of-atmosphere reflectance.** It is what the
sensor measured, atmosphere included. Production pipelines normally use Level-2A
(surface reflectance, atmospherically corrected), and that is also what the AWS
Earth Search catalogue serves. **A model trained on L1C and applied to L2A is
operating under a real covariate shift.** Flagging it here; notebook 05
quantifies it, mitigates it, and reports what it cost.

**3. One pixel is 10 m across, and that dictates the task.** A car is 4 m, a
house is 10 m. At this ground sampling distance, object detection of small
objects is not a hard problem, it is a physically impossible one — the object is
subpixel and the information is simply not present. The tasks the sensor
actually supports are classification, semantic segmentation, and change
detection over time. This is why the project is a classification-and-mapping
study rather than a detection one: it is a judgement about the instrument, not a
preference about models.

A related consequence worth keeping in mind for notebook 06: a 64×64 chip covers
640×640 m. That is a large area to summarise with one label — a chip containing
a road, some houses and a field genuinely has no single correct answer.

### Class distribution

Whether the dataset is balanced determines whether accuracy is a defensible
headline metric. EuroSAT is *roughly* balanced (2,000–3,000 chips per class),
which means accuracy will not be badly misleading — but the classes still differ
by 50% between the largest and smallest, so we report **macro-F1 alongside
accuracy everywhere**. Macro-F1 weights each class equally, so a model that
quietly gives up on `Pasture` is penalised in a way accuracy would hide.

In [ ]:
from s2map import viz as V

V.set_style()
counts = np.bincount(y, minlength=len(C.CLASS_NAMES))
print("largest / smallest class ratio:", round(counts.max() / counts.min(), 2))

fig = V.plot_class_distribution(y)
V.save(fig, "01_class_distribution")

### Rendering, part 1: why raw satellite values look black

This is the small thing that separates people who have handled satellite imagery
from people who have not.

Reflectance over land is typically 0.02–0.35 of incident light, stored here as
integers scaled by 10,000, while the array's dynamic range runs far higher (a
bright cloud or specular water glint can saturate). Rendering the raw values
directly maps almost the entire land surface into the bottom few percent of the
display range, and you get a black square.

The fix is a **percentile stretch**: take the 2nd and 98th percentile of each
band, map that interval linearly onto [0, 1], clip the rest. The outliers that
dominated the range are discarded and the actual scene fills the display. Doing
it per band also removes the blue cast that Rayleigh scattering (strongest at
short wavelengths) puts on every uncorrected optical satellite image.

The stretch is a *display* transform only. It never touches the arrays the
models see — those keep physical units.

In [ ]:
example_idx = int(np.flatnonzero(y == C.CLASS_NAMES.index("Residential"))[0])
chip = np.asarray(X[example_idx])
print("chip", chip.shape, chip.dtype, "min", chip.min(), "max", chip.max())

fig = V.plot_stretch_comparison(chip, class_name="Residential")
V.save(fig, "01_percentile_stretch")

### Rendering, part 2: one chip per class, true colour

A first look at what the ten classes are. Note how little separates
`Residential` from `Industrial` visually at 10 m, and how similar the four
vegetation classes look — that similarity is the thing the spectral analysis
below turns into a testable prediction.

In [ ]:
rng = np.random.default_rng(cfg.seed)
one_per_class = [int(rng.choice(np.flatnonzero(y == c))) for c in range(len(C.CLASS_NAMES))]
chips = [np.asarray(X[i]) for i in one_per_class]

fig = V.plot_chip_grid(chips, C.CLASS_NAMES, B.true_color, n_cols=5,
                       title="EuroSAT classes, true colour (B04/B03/B02, 2–98% stretch)")
V.save(fig, "01_true_color_grid")

### Rendering, part 3: false colour makes the invisible bands visible

Two standard composites, each putting non-visible bands into the display's red,
green and blue guns:

- **NIR composite (B08/B04/B03).** Healthy vegetation appears vivid red:
  chlorophyll absorbs red light while the internal structure of leaves scatters
  near-infrared strongly, so a leaf is dark in B04 and bright in B08. Vegetation
  vigour becomes visually obvious, and water goes near-black.
- **SWIR composite (B12/B08/B04).** Shortwave infrared is absorbed by water, so
  this composite tracks moisture: wet soil, burnt ground and bare rock separate
  clearly, and built-up surfaces brighten.

Same chip, three renderings — the panel below is the argument for why we kept
all thirteen bands.

In [ ]:
for class_name in ("Forest", "Industrial", "SeaLake"):
    i = int(rng.choice(np.flatnonzero(y == C.CLASS_NAMES.index(class_name))))
    fig = V.plot_composites(np.asarray(X[i]), class_name=class_name)
    V.save(fig, f"01_composites_{class_name.lower()}")

### Spectral signatures — the key figure of this notebook

For each class we average every band over many chips and plot the resulting
curve against wavelength. This is the class's *spectral signature*, and it is
the physical reason classification works at all.

We subsample chips per class rather than using all 27,000: the curves are means
over ~800 × 4,096 = 3.3 million pixels per class, which is already far more than
needed for a stable mean, and it keeps this cell under a minute on a memmapped
array.

In [ ]:
N_PER_CLASS = 800
means, stds = [], []
for c in range(len(C.CLASS_NAMES)):
    idx = np.flatnonzero(y == c)
    idx = np.sort(rng.choice(idx, size=min(N_PER_CLASS, idx.size), replace=False))
    block = np.asarray(X[idx], dtype=np.float64)          # (n, 13, 64, 64)
    per_band = block.transpose(1, 0, 2, 3).reshape(13, -1)
    means.append(per_band.mean(axis=1))
    stds.append(per_band.std(axis=1))

means, stds = np.stack(means), np.stack(stds)
print("signature matrix", means.shape, "(classes x bands)")
assert means.shape == (10, 13)

fig = V.plot_spectral_signatures(means, stds)
V.save(fig, "01_spectral_signatures")

### Reading the signatures — and a prediction we will test in notebook 02

Three structures should be visible:

- **The vegetation red edge.** `Forest`, `Pasture`, `HerbaceousVegetation` and
  `AnnualCrop` all dip at B04 (665 nm, chlorophyll absorption) and jump sharply
  into B08 (842 nm). That jump is the single most useful feature in optical
  remote sensing, and it is what NDVI measures.
- **Water absorbs infrared.** `SeaLake` and `River` collapse towards zero across
  B08 and both SWIR bands — liquid water absorbs almost all radiation past
  ~900 nm. Water is therefore the easiest class in the dataset.
- **Built-up surfaces are spectrally flat and bright.** `Residential` and
  `Industrial` sit high across the visible and SWIR with no red edge, because
  concrete, tile and asphalt have no chlorophyll.

**The prediction.** The four vegetation classes share a signature shape and are
separated only by amplitude, which is exactly the kind of difference that
illumination, season and soil background also produce. So:

> The dominant confusions in notebook 02 will be *within* the vegetation group —
> AnnualCrop ↔ PermanentCrop ↔ HerbaceousVegetation ↔ Pasture — and if anything
> spectral separates them it will be the SWIR bands (moisture / soil exposure),
> not the visible ones. Water classes will be near-perfect. `Highway` and
> `River` will be confused with whatever surrounds them, because a 640 m chip
> containing a thin linear feature is mostly *not* that feature.

Notebook 02 checks this against the confusion matrix, and notebook 06 checks it
a third way with a band-ablation study. Closing that loop is the point.

### Spectral indices: features from physics, with no training at all

Three normalised band ratios, each isolating one surface property:

| Index | Formula | Responds to |
|---|---|---|
| NDVI | (B08 − B04) / (B08 + B04) | green biomass / vegetation vigour |
| NDWI | (B03 − B08) / (B03 + B08) | open water |
| NDBI | (B11 − B08) / (B11 + B08) | built-up / impervious surfaces |

They are ratios, which is the point: a ratio cancels multiplicative effects such
as illumination angle and gain, so NDVI over a shaded slope is much closer to
NDVI over a sunlit one than the raw bands are. They are also the lowest rung of
the model ladder in this project — zero parameters, zero labels, pure physics —
and notebook 02's Arm 0 measures how far they alone get you.

Division by zero is real here (both bands read zero over no-data), so
`normalized_difference` guards it and returns the neutral value 0.

In [ ]:
SUB = 4000
sub_idx = np.sort(rng.choice(len(y), size=SUB, replace=False))
sub = np.asarray(X[sub_idx], dtype=np.float64)
index_values = B.spectral_index_features(sub)     # (n, 3) mean NDVI/NDWI/NDBI per chip

print("index features", index_values.shape,
      "min", np.round(index_values.min(0), 3), "max", np.round(index_values.max(0), 3))
assert np.all(np.isfinite(index_values)), "index computation produced non-finite values"
assert index_values.min() >= -1.0 and index_values.max() <= 1.0

fig = V.plot_index_boxplots(index_values, y[sub_idx])
V.save(fig, "01_spectral_indices")

for k, name in enumerate(("NDVI", "NDWI", "NDBI")):
    order = np.argsort([index_values[y[sub_idx] == c, k].mean() for c in range(10)])
    print(f"{name:>5} class order (low to high): "
          + " < ".join(C.CLASS_NAMES[c] for c in order))

### Splits — and an honest note about why this one is imperfect

Stratified 70/15/15, fixed seed, saved to `outputs/splits.npz` as index arrays so
every later notebook uses **the identical split**. Stratification means each
class is split by the same proportions, so the val and test sets have the same
class balance as the whole; without it, a rare class can end up with a handful of
test examples and its F1 becomes noise.

**The limitation, stated up front.** EuroSAT's 27,000 chips are cut from a
limited number of Sentinel-2 scenes, so chips are spatially autocorrelated: two
chips from the same field, minutes apart in the same overpass, can land in train
and in test. A random split therefore leaks information and mildly overstates
every number reported in this project. The methodologically correct fix is a
**spatially blocked split** — group by source scene or geographic region and hold
out whole blocks — which is standard practice in geospatial ML and routinely
skipped.

We keep the random split anyway, for one reason: comparability. Published EuroSAT
numbers use it, and a project whose numbers cannot be compared to anything is
less useful than one with a known, documented bias. This limitation is repeated
in the README rather than buried here.

In [ ]:
splits = D.make_stratified_splits(
    y, cfg.split.train_frac, cfg.split.val_frac, cfg.split.test_frac, cfg.split.seed
)
for name, idx in splits.items():
    print(f"{name:<6} {idx.size:>6,} chips  ({idx.size / y.size:.1%})")

# leakage guard, asserted here and unit-tested in tests/test_data.py
train, val, test = (set(splits[k].tolist()) for k in ("train", "val", "test"))
assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
assert len(train | val | test) == y.size

props = {k: np.bincount(y[v], minlength=10) / v.size for k, v in splits.items()}
overall = np.bincount(y, minlength=10) / y.size
print("\nmax deviation of any class proportion from the overall rate: "
      f"{max(np.abs(p - overall).max() for p in props.values()):.4f}")

path = D.save_splits(splits)
print("saved", path)

### Normalisation statistics — computed on the training split only

Per-band mean and standard deviation, used to standardise inputs for every
trained model in the project.

**Computed on the training indices only, never the full dataset.** Using all
27,000 chips would leak test-set statistics into the preprocessing: the model's
inputs would be shaped, however slightly, by data it is supposed to have never
seen. On a near-balanced dataset the numerical effect is small, but it costs
nothing to do correctly, and getting it wrong is one of the first things a
careful reviewer checks.

Computed in chunks so the full 2.9 GB array is never materialised.

In [ ]:
stats = D.compute_band_stats(X, splits["train"])
path = D.save_band_stats(stats)
print("saved", path, "\n")
print(f"{'band':<6}{'mean':>12}{'std':>12}")
for b, m, s in zip(stats["band_ids"], stats["mean"], stats["std"]):
    print(f"{b:<6}{m:>12.1f}{s:>12.1f}")
print(f"\ncomputed over {stats['n_pixels']:,} pixels ({stats['computed_on']})")

# Sanity check 1: standardising the training data with these statistics must
# give approximately N(0, 1).
#
# The sample is drawn RANDOMLY across the training split, not as
# splits["train"][:256]. The split indices are sorted and the underlying array
# is grouped by class, so the first 256 training chips are all AnnualCrop —
# one class, whose mean is legitimately offset from the global mean. Checking
# against that slice fails while the statistics are perfectly correct. "The
# first N rows" is not a sample of a sorted array; this is a small instance of
# the same assumption that makes people trust a random split on spatially
# autocorrelated data.
rng_check = np.random.default_rng(cfg.seed)
check_idx = np.sort(rng_check.choice(splits["train"], size=2048, replace=False))
check = D.normalize(np.asarray(X[check_idx]), stats)
print(f"\ncheck on 2,048 random training chips: mean {check.mean():+.4f}  std {check.std():.4f}")
assert abs(check.mean()) < 0.05 and abs(check.std() - 1) < 0.10

# For contrast, the class-ordered slice that looks like a sample and is not:
biased = D.normalize(np.asarray(X[splits["train"][:256]]), stats)
print(f"the first 256 training chips ({C.CLASS_NAMES[y[splits['train'][0]]]} only): "
      f"mean {biased.mean():+.4f}  std {biased.std():.4f}  <- not a sample, do not check against this")

### Independent check: does the band order match published EuroSAT statistics?

The band statistics just computed can be compared against the per-band means
published with EuroSAT (as distributed in torchgeo's loader). This is not about
the exact numbers — ours come from the training split only, so they will differ
slightly and that is meaningless.

**It is a test of band order**, the one bug in this pipeline that raises nothing
and invalidates everything downstream. If a mirror silently permuted the bands,
or handed back a different product, our means would not track the reference. The
signature is unusually sharp here because EuroSAT's B09 mean is ~12 while its
neighbours are in the thousands — a permutation cannot hide that.

In [ ]:
ok, report = B.check_band_order(stats["mean"], tolerance=0.10)
print(f"{'band':<6}{'computed':>10}{'published':>12}{'rel.dev':>10}")
for line in report:
    print(line)
print(f"\nband order verified: {ok}")
assert ok, ("computed band means deviate from published EuroSAT statistics — "
            "the bands are probably in the wrong order, which would silently "
            "corrupt every result in this project")

### Record what this notebook established

In [ ]:
from s2map import evaluate as E

E.append_result({
    "notebook": "01",
    "arm": "dataset_summary",
    "input": "13-band EuroSAT",
    "n_samples": int(y.size),
    "n_classes": int(len(C.CLASS_NAMES)),
    "class_counts": {n: int(c) for n, c in zip(C.CLASS_NAMES, counts)},
    "split_sizes": {k: int(v.size) for k, v in splits.items()},
    "source": bundle.source,
    "notes": "L1C top-of-atmosphere; random (not spatially blocked) split",
})
print("results.json updated")

## Findings

1. **The physics is already doing most of the work.** The ten classes separate
   into three spectrally distinct families — vegetation (red edge at B04→B08),
   water (near-zero NIR/SWIR), and built-up (flat and bright) — before any model
   is trained. Any classifier that works here is largely rediscovering this.

2. **The hard problem is *within* the vegetation family.** AnnualCrop,
   PermanentCrop, HerbaceousVegetation and Pasture share a signature shape and
   differ mainly in amplitude. This generates the concrete prediction, tested in
   notebooks 02 and 06, that these four classes dominate the confusion matrix.

3. **Ten of the thirteen bands are invisible to any RGB model.** The red edge,
   NIR and SWIR bands carry the separability. This is stated here so that
   notebook 03's CLIP results — where those ten bands must be discarded — are
   read as a structural handicap rather than a surprise.

4. **Rendering satellite data requires a percentile stretch**, and the
   before/after figure shows why: without it the imagery is black, because land
   reflectance occupies a small low slice of the value range.

5. **The split we use is knowingly imperfect.** Spatial autocorrelation between
   EuroSAT chips means a random split leaks, and every number in this project is
   therefore mildly optimistic. Spatially blocked splitting is the correct fix;
   we trade it for comparability with published numbers and say so.

**Next:** notebook 02 builds the supervised ladder — classical features, a small
CNN, ResNet-18, ViT — and tests the confusion prediction made above.